In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("IcebergSparkWithMinIo") \
        .config("spark.sql.catalog.iceberg_catalog", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.iceberg_catalog.type", "rest") \
        .config("spark.sql.catalog.iceberg_catalog.uri", "http://iceberg-rest:8181") \
        .config("spark.sql.catalog.iceberg_catalog.warehouse", "s3://warehouse/") \
        .config("spark.sql.catalog.iceberg_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
        .config("spark.sql.catalog.iceberg_catalog.s3.endpoint", "http://minio:9000") \
        .config("spark.sql.catalog.iceberg_catalog.s3.access-key-id", "admin") \
        .config("spark.sql.catalog.iceberg_catalog.s3.secret-access-key", "password") \
        .config("spark.sql.catalog.iceberg_catalog.s3.path-style-access", "true") \
        .getOrCreate()

spark.sql("SHOW CATALOGS").show()

+---------------+
|        catalog|
+---------------+
|           demo|
|iceberg_catalog|
|  spark_catalog|
+---------------+



In [13]:
spark.sql("SHOW DATABASES IN iceberg_catalog").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [14]:
spark.sql("CREATE DATABASE IF NOT EXISTS iceberg_catalog.default")

DataFrame[]

In [6]:
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg_catalog.default.my_table (
    id BIGINT,
    name STRING
) USING iceberg
""")

DataFrame[]

In [7]:
spark.sql("""
SELECT * FROM iceberg_catalog.default.my_table
""").show()

+---+---------+
| id|     name|
+---+---------+
|  1|KIMJAEMIN|
|  2|  KIMTEST|
+---+---------+



In [11]:
spark.sql("""
INSERT INTO iceberg_catalog.default.my_table
VALUES (1, 'KIMJAEMIN'), (2, 'KIMTEST')
""")

DataFrame[]

In [8]:
spark.sql("""
UPDATE iceberg_catalog.default.my_table SET name = 'CHOITEST' WHERE id = 2
""")

DataFrame[]

In [9]:
spark.sql("""
SELECT * FROM iceberg_catalog.default.my_table
""").show()

+---+---------+
| id|     name|
+---+---------+
|  2| CHOITEST|
|  1|KIMJAEMIN|
+---+---------+



In [24]:
spark.sql("""
SELECT * FROM iceberg_catalog.default.my_table.snapshots
ORDER BY committed_at
""").show(truncate=True)

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-10-05 15:17:...|1639227104772827266|               NULL|   append|s3://warehouse/de...|{spark.app.id -> ...|
|2025-10-05 16:52:...|2656649516270997423|1639227104772827266|overwrite|s3://warehouse/de...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+

